# Roman-to-Roman Transliteration

The following Jupyter notebook exhibits the 12 steps given in
the algorithm described in [Tafseer Ahmed's paper](https://www.cle.org.pk/clt09/download/ahmed_translit.pdf).
The accompanying description is verbetim from the aforemtnioned article.

In [1]:
import re

from romanalfaz.algorithm import (
    permuteAllOccurrences, replaceEnding, permuteAllEndings, permuteConsecutiveVowels
)

## Example Words
The following seven example words will be walked through on each step to showcase
the transformation at each stage.

In [2]:
rmWords00 = [
    'alag',
    'ullo',
    'bukhar',
    'bhai',
    'hai',
    'bhayi',
    'shohrat',
    'kya',
]

for i, w in enumerate(rmWords00, start=1):
    print(f'{i:02}. "{w}"')

01. "alag"
02. "ullo"
03. "bukhar"
04. "bhai"
05. "hai"
06. "bhayi"
07. "shohrat"
08. "kya"


## Step 1
Except `a`, `e`, `i`, `o`, `u`, `y` and `h`, change the
case of all the characters of `rom_word` into capital.
This transformed encoded word is termed as
`enc_rom_word`.

### Explanation
As vowel mapping need more
complex processing than one to one replacement, the
encoding is applied on consonants only.
A character in capital case means a rule is already
applied on it and the following low priority rule will
not be accidentally applied on it.

### Example Words
- `aLaG`
- `uLLo`
- `BuKhaR`
- `Bhai`
- `hai`
- `Bhayi`
- `ShohRaT`

In [3]:
def step01(rmWord: str) -> str:
    """Vowel separation"""
    vowels = ['a', 'e', 'i', 'o', 'u', 'y', 'h']
    transWord = ''
    for ch in rmWord:
        transWord += ch.upper() if ch not in vowels else ch
    return transWord

In [4]:
rmWords01 = [step01(w) for w in rmWords00]

for i, (w1, w2) in enumerate(zip(rmWords00, rmWords01), start=1):
    print(f'{i:02}. "{w1}" -> "{w2}"')

01. "alag" -> "aLaG"
02. "ullo" -> "uLLo"
03. "bukhar" -> "BuKhaR"
04. "bhai" -> "Bhai"
05. "hai" -> "hai"
06. "bhayi" -> "Bhayi"
07. "shohrat" -> "ShohRaT"
08. "kya" -> "Kya"


## Step 2
If the two consequent capital letters are the same,
delete one of those double letters.

### Explanation
The rule deals the
germination/tashdeed as explained in 2.4. it deletes one
of the double consonants because the germinated
consonant is written only once in Urdu script.

### Example Words
- `aLaG`
- `uLo`
- `BuKhaR`
- `Bhai`
- `hai`
- `Bhayi`
- `ShohRaT`

In [5]:
def step02(rmWord: str) -> str:
    """Tashdeed - Remove double consonants"""
    return re.sub(r'([A-Z])\1+', r'\1', rmWord)

In [6]:
rmWords02 = [step02(w) for w in rmWords01]

for i, (w1, w2) in enumerate(zip(rmWords01, rmWords02), start=1):
    print(f'{i:02}. "{w1}" -> "{w2}"')

01. "aLaG" -> "aLaG"
02. "uLLo" -> "uLo"
03. "BuKhaR" -> "BuKhaR"
04. "Bhai" -> "Bhai"
05. "hai" -> "hai"
06. "Bhayi" -> "Bhayi"
07. "ShohRaT" -> "ShohRaT"
08. "Kya" -> "Kya"


## Step 3
If the word begins with a vowel (any roman letter
or letter sequence present in Table 2), append `A` at the
beginning of the word.

### Explanation
We need an
extra `a` sound character before a vowel at the start of
the word in Urdu script. For this purpose, we introduce
`A` in `enc_rom_word` that will get matched with Urdu
equivalent. For example, Urdu word اينٹ 'brick' is
written as `eent` in roman script. In the roman word,
`ee` stands for Urdu "ی" but we need to put a "ا" at the
start.

### Example Words
- `AaLaG`
- `AuLo`
- `BuKhaR`
- `Bhai`
- `hai`
- `Bhayi`
- `ShohRaT`

In [7]:
def step03(rmWord: str) -> str:
    """Add alef to words starting with vowel"""
    vowels = ['a', 'e', 'i', 'o', 'u']
    transWord = 'A' if rmWord[0] in vowels else ''
    transWord += rmWord
    return transWord

In [8]:
rmWords03 = [step03(w) for w in rmWords02]

for i, (w1, w2) in enumerate(zip(rmWords02, rmWords03), start=1):
    print(f'{i:02}. "{w1}" -> "{w2}"')

01. "aLaG" -> "AaLaG"
02. "uLo" -> "AuLo"
03. "BuKhaR" -> "BuKhaR"
04. "Bhai" -> "Bhai"
05. "hai" -> "hai"
06. "Bhayi" -> "Bhayi"
07. "ShohRaT" -> "ShohRaT"
08. "Kya" -> "Kya"


## Step 4
For the sequences `eh` and `oh`, do the following
replacements. Consider the longest match at left hand
side.
```
ehe = eHe, H
eh = eH, H
oh = oH, H
h = H
```

### Explanation
In our mapping rules, if there are
more than one potential replacements at right hand side
corresponding to a single left hand side, then the
system makes `n` number of copies of `enc_rom_word`.
Each of the possible right hand side replacement is
applied on one of the copies, and the subsequent steps
of algorithm are applied on each of those.
The above mapping rules deal with the unwritten
long vowel before a "ہ" (gol-hay) in Urdu script. It is
discussed in 2.5.

### Example Words
- `AaLaG`
- `AuLo`
- `BuKHaR`
- `BHai`
- `Hai`
- `BHayi`
- `SHoHRaT` / `SHHRaT`

In [9]:
hVowelCombos = {
    'ehe': ['eHe', 'H'],
    'eh': ['eH', 'H'],
    'oh': ['oH', 'H'],
    'h': ['H'],
}

def step04(rmWord: str) -> list:
    """Process eh and oh vowels"""
    return permuteAllOccurrences([rmWord], hVowelCombos)

In [10]:
rmWords04 = [step04(w) for w in rmWords03]

for i, (w1, w2) in enumerate(zip(rmWords03, rmWords04), start=1):
    print(f'{i:02}. "{w1}" -> {w2}')

01. "AaLaG" -> ['AaLaG']
02. "AuLo" -> ['AuLo']
03. "BuKhaR" -> ['BuKHaR']
04. "Bhai" -> ['BHai']
05. "hai" -> ['Hai']
06. "Bhayi" -> ['BHayi']
07. "ShohRaT" -> ['SHoHRaT', 'SHHRaT']
08. "Kya" -> ['Kya']


## Step 5
If `y` is the last character of the word and is
preceded by `e` or `a`, then
```
ey = Y
ay = E
```

### Explanation
As discussed in 2.8, both chooti-ye
and bari-ye are written as "ی" (chooti-ye) `Y` in medial
position, but bari-ye is written as "ے" (bari-ye) `E`
only at the final position.

### Example Words
- `AaLaG`
- `AuLo`
- `BuKHaR`
- `BHai`
- `Hai`
- `BHayi`
- `SHoHRaT` / `SHHRaT`

In [11]:
def step05(rmWords: list[str]) -> list[str]:
    """Process ending yeh"""
    yehEndings = {
        'ey': 'Y',
        'ay': "E",
    }
    return [replaceEnding(rmWord, yehEndings, 2) for rmWord in rmWords]

In [12]:
rmWords05 = [step05(ww) for ww in rmWords04]

for i, (w1, w2) in enumerate(zip(rmWords04, rmWords05), start=1):
    print(f'{i:02}. {w1} -> {w2}')

01. ['AaLaG'] -> ['AaLaG']
02. ['AuLo'] -> ['AuLo']
03. ['BuKHaR'] -> ['BuKHaR']
04. ['BHai'] -> ['BHai']
05. ['Hai'] -> ['Hai']
06. ['BHayi'] -> ['BHayi']
07. ['SHoHRaT', 'SHHRaT'] -> ['SHoHRaT', 'SHHRaT']
08. ['Kya'] -> ['Kya']


## Step 6
If `y` is preceded by `e` or `a` and followed by a vowel then
```
ey = Y, eY
ay = Y, aY
```

### Explanation
The `y` in this case can act either as
consonant or as part of a vowel sequence. For example,
in `gayi` گئ "(she) went", `y` acts as consonant. Step 8
gives more details about the rules of this type.

### Example Words
- `AaLaG`
- `AuLo`
- `BuKHaR`
- `BHai`
- `Hai`
- `BHYi` / `BHaYi`
- `SHoHRaT` / `SHHRaT`

In [13]:
yVowelCombos = {
    'ey': ['Y', 'eY'],
    'ay': ['Y', 'aY'],
}

def step06(rmWords: list) -> list:
    """Process ey and ay vowels"""
    return permuteAllOccurrences(rmWords, yVowelCombos)

In [14]:
rmWords06 = [step06(ww) for ww in rmWords05]

for i, (w1, w2) in enumerate(zip(rmWords05, rmWords06), start=1):
    print(f'{i:02}. {w1} -> {w2}')

01. ['AaLaG'] -> ['AaLaG']
02. ['AuLo'] -> ['AuLo']
03. ['BuKHaR'] -> ['BuKHaR']
04. ['BHai'] -> ['BHai']
05. ['Hai'] -> ['Hai']
06. ['BHayi'] -> ['BHYi', 'BHaYi']
07. ['SHoHRaT', 'SHHRaT'] -> ['SHoHRaT', 'SHHRaT']
08. ['Kya'] -> ['Kya']


## Step 7
Change the case of `y` as capital.
```
y = Y
```

### Explanation
As we have dealt with all the special
cases of `y`, this general rule changes the case of the
remaining ones as capital.

### Example Words
- `AaLaG`
- `AuLo`
- `BuKHaR`
- `BHai`
- `Hai`
- `BHYi` / `BHaYi`
- `SHoHRaT` / `SHHRaT`

In [15]:
def step07(rmWords: list[str]) -> list:
    """Process y consonants to Y."""
    return [w.replace('y', 'Y') for w in rmWords]

In [16]:
rmWords07 = [step07(ww) for ww in rmWords06]

for i, (w1, w2) in enumerate(zip(rmWords06, rmWords07), start=1):
    print(f'{i:02}. {w1} -> {w2}')

01. ['AaLaG'] -> ['AaLaG']
02. ['AuLo'] -> ['AuLo']
03. ['BuKHaR'] -> ['BuKHaR']
04. ['BHai'] -> ['BHai']
05. ['Hai'] -> ['Hai']
06. ['BHYi', 'BHaYi'] -> ['BHYi', 'BHaYi']
07. ['SHoHRaT', 'SHHRaT'] -> ['SHoHRaT', 'SHHRaT']
08. ['Kya'] -> ['KYa']


## Step 8
If the vowel sequence `ai` or `ei` is present at the
end of the word, then apply following replacement.
```
ai = E, aYi, aAi
ei = E, eYi, e
```

### Explanation
As discussed in 2.10, the character
sequence `ai` either correspond to single letter "ے"
(bari-ye) or it is a sequence of two vowels in two
different syllables. In this case, `a` is in the first
syllable and `i` is the start of the second syllable. As
we need "ء" (hamza) or "ع" (ain) before the vowel at
syllable initial position, `Y` and `A` are introduced to
represent these characters

### Example Words
- `AaLaG`
- `AuLo`
- `BuKHaR`
- `BHE` / `BHaYi` / `BHaAi`
- `HE` / `HaYi` / `HaAi`
- `BHYi` / `BHaYi`
- `SHoHRaT` / `SHHRaT`

In [17]:
def step08(rmWords: list[str]) -> list[str]:
    """Process ending of ai and ei"""
    iEndings = {
        'ai': ['E', 'aYi', 'aAi'],
        'ei': ['E', 'eYi', 'eAi'],
    }
    return permuteAllEndings(rmWords, iEndings)

In [18]:
rmWords08 = [step08(ww) for ww in rmWords07]

for i, (w1, w2) in enumerate(zip(rmWords07, rmWords08), start=1):
    print(f'{i:02}. {w1} -> {w2}')

01. ['AaLaG'] -> ['AaLaG']
02. ['AuLo'] -> ['AuLo']
03. ['BuKHaR'] -> ['BuKHaR']
04. ['BHai'] -> ['BHE', 'BHaYi', 'BHaAi']
05. ['Hai'] -> ['HE', 'HaYi', 'HaAi']
06. ['BHYi', 'BHaYi'] -> ['BHYi', 'BHaYi']
07. ['SHoHRaT', 'SHHRaT'] -> ['SHoHRaT', 'SHHRaT']
08. ['KYa'] -> ['KYa']


## Step 9
If there is a sequence (`seq`) of two or more vowels,
then find all the combinations of valid vowel sequences
`seq1`, `seq2`, ..., `seqn`, and put `A` for "ع" (ain) or `Y`
for "ء" (hamza) is put between these valid sequences.

### Explanation
This rule is a generalized form of
rule 8. It deals with all possible interpretations of
sequence of vowels. An example of two letter
sequences is `ai` that has two valid vowel combinations
`a-i` and `ai` (as given in table 2). By applying the rule,
we get `aAi`, `aYi` and `ai` for further processing.

Another two letter sequence is `ua` that has only
one valid combination `u-a`. The other possibility `ua`
is not a valid vowel combination because `ua` does not
map on any single Urdu vowel. Hence, we get `uYa`
and `uAa` for further processing in subsequent steps.

An example of three vowel letters in a row is `aai` آئ '(she) came'.
It has three valid sequences `a-ai`, `aai`, `a-a-i`.
By applying the rule, we get `aAai`, `aYai`,
`aaAi`, `aaYi`, `aAaAi`, `aAaYi`, `aYaYi` and `aYaAi`
for further processing.

### Example Words
- `AaLaG`
- `AuLo`
- `BuKHaR`
- `BHE` / `BHaYi` / `BHaAi`
- `HE` / `HaYi` / `HaAi`
- `BHYi` / `BHaYi`
- `SHoHRaT` / `SHHRaT`

In [19]:
def step09(rmWords: list[str]) -> list[str]:
    """Process two or more consecutive vowels"""
    return permuteConsecutiveVowels(rmWords)

In [20]:
samples = ["sait", "ua", "aai"]
for w in samples:
    print(f'{w} -> {step09([w])}')
print('')

rmWords09 = [step09(ww) for ww in rmWords08]

for i, (w1, w2) in enumerate(zip(rmWords08, rmWords09), start=1):
    print(f'{i:02}. {w1} -> {w2}')

sait -> ['saAit', 'saYit', 'sait']
ua -> ['uAa', 'uYa']
aai -> ['aAaAi', 'aAaYi', 'aYaAi', 'aYaYi', 'aAai', 'aYai', 'aaAi', 'aaYi']

01. ['AaLaG'] -> ['AaLaG']
02. ['AuLo'] -> ['AuLo']
03. ['BuKHaR'] -> ['BuKHaR']
04. ['BHE', 'BHaYi', 'BHaAi'] -> ['BHE', 'BHaYi', 'BHaAi']
05. ['HE', 'HaYi', 'HaAi'] -> ['HE', 'HaYi', 'HaAi']
06. ['BHYi', 'BHaYi'] -> ['BHYi', 'BHaYi']
07. ['SHoHRaT', 'SHHRaT'] -> ['SHoHRaT', 'SHHRaT']
08. ['KYa'] -> ['KYa']


## Step 10
For two vowel sequence, do the following
replacements.
```
aa = A
ai = Y
ei = Y
ee = Y
ie = Y
oo = O
au = O
ou = O
```

### Explanation
It is simple one to one mapping of
vowel sequence with encoding of corresponding Urdu
letter.

### Example Words
- `AaLaG`
- `AuLo`
- `BuKHaR`
- `BHE` / `BHaYi`
- `HE` / `HaYi` / `HaAi`
- `BHYi` / `BHaYi` / `BHaAi`
- `SHoHRaT` / `SHHRaT`

In [21]:
DoubleVowels = {
    'aa': ['A'],
    'ai': ['Y'],
    'ei': ['Y'],
    'ee': ['Y'],
    'ie': ['Y'],
    'oo': ['O'],
    'au': ['O'],
    'ou': ['O'],
}
def step10(rmWords: list[str]) -> list:
    """Process two vowel sequences."""
    return permuteAllOccurrences(rmWords, DoubleVowels)

In [22]:
rmWords10 = [step10(ww) for ww in rmWords09]

for i, (w1, w2) in enumerate(zip(rmWords09, rmWords10), start=1):
    print(f'{i:02}. {w1} -> {w2}')

01. ['AaLaG'] -> ['AaLaG']
02. ['AuLo'] -> ['AuLo']
03. ['BuKHaR'] -> ['BuKHaR']
04. ['BHE', 'BHaYi', 'BHaAi'] -> ['BHE', 'BHaYi', 'BHaAi']
05. ['HE', 'HaYi', 'HaAi'] -> ['HE', 'HaYi', 'HaAi']
06. ['BHYi', 'BHaYi'] -> ['BHYi', 'BHaYi']
07. ['SHoHRaT', 'SHHRaT'] -> ['SHoHRaT', 'SHHRaT']
08. ['KYa'] -> ['KYa']


## Step 11
Search the following vowels at word’s final position and make the substitutions accordingly.
```
e = E
a = A, H
i = Y
u = O
```

### Explanation
In Urdu script, the word’s final
vowel is always written as long vowel. For example the
final `i` of _aadmi_ 'man' is not ambiguous between
short vowel (unwritten diacratic _zer_) and long vowel
(chooti-ye). Only chooti-ye can appear at the end of
any word.

The final `a` can map on "ہ" (gol-hay) too. For
example, the final `a` of roman _sada_ سادہ 'simple'
stands for gol-hay.

### Example Words
- `AaLaG`
- `AuLO`
- `BuKHaR`
- `BHE` / `BHaYY` / `BHaAY`
- `HE` / `HaYY` / `HaAY`
- `BHYY` / `BHaYY`
- `SHoHRaT` / `SHHRaT`

In [27]:
VowelEndings = {
    'e': ['E'],
    'a': ['A', 'H'],
    'i': ['Y'],
    'u': ['O']
}

def step11(rmWords: list[str]) -> list:
    """Process two vowel sequences."""
    return permuteAllEndings(rmWords, VowelEndings)

In [28]:
rmWords11 = [step11(ww) for ww in rmWords10]

for i, (w1, w2) in enumerate(zip(rmWords10, rmWords11), start=1):
    print(f'{i:02}. {w1} -> {w2}')

01. ['AaLaG'] -> ['AaLaG']
02. ['AuLo'] -> ['AuLo']
03. ['BuKHaR'] -> ['BuKHaR']
04. ['BHE', 'BHaYi', 'BHaAi'] -> ['BHE', 'BHaYY', 'BHaAY']
05. ['HE', 'HaYi', 'HaAi'] -> ['HE', 'HaYY', 'HaAY']
06. ['BHYi', 'BHaYi'] -> ['BHYY', 'BHaYY']
07. ['SHoHRaT', 'SHHRaT'] -> ['SHoHRaT', 'SHHRaT']
08. ['KYa'] -> ['KYA', 'KYH']


## Step 12
Search for the following vowel sequences and make
the following replacements.
```
a = null, A
i = null, Y
u = null, O
e = E
o = O
```

### Explanation
After dealing the vowels at initial
and final positions, and dealing the special cases of
vowel sequence, the general rule of vowel sequence
replacement is given.

This is the last step for encoding a roman word. The
following steps search the equivalent of the encoded
word in the encoded list of Urdu words.

### Example Words
- `AALAG` / `ALAG` / `AALG` / `ALG`
- `AOLO` / `ALO`
- `BOKHAR` / `BKHAR` / `BOKHR` / `BKHR`
- `BHE` / `BHAYY` / `BHYY` / `BHAAY` / `BHAY`
- `HE` / `HAYY`/ `HYY` / `HAAY` / `HAY`
- `BHYY` / `BHAYY` / `BHYY`
- `SHOHRAT`/ `SHOHRT` / `SHHRAT` / `SHHRT`

In [25]:
VowelReplacements = {
    'a': ['', 'A'],
    'i': ['', 'Y'],
    'u': ['', 'O'],
    'e': ['E'],
    'o': ['O'],
}

def step12(rmWords: list[str]) -> list:
    """Process two vowel sequences."""
    return permuteAllOccurrences(rmWords, VowelReplacements)

In [26]:
rmWords12 = [step12(ww) for ww in rmWords11]

for i, (w1, w2) in enumerate(zip(rmWords11, rmWords12), start=1):
    print(f'{i:02}. {w1} -> {w2}')

01. ['AaLaG'] -> ['ALG', 'ALAG', 'AALG', 'AALAG']
02. ['AuLo'] -> ['ALO', 'AOLO']
03. ['BuKHaR'] -> ['BKHR', 'BKHAR', 'BOKHR', 'BOKHAR']
04. ['BHE', 'BHaYY', 'BHaAY'] -> ['BHE', 'BHYY', 'BHAYY', 'BHAY', 'BHAAY']
05. ['HE', 'HaYY', 'HaAY'] -> ['HE', 'HYY', 'HAYY', 'HAY', 'HAAY']
06. ['BHYY', 'BHaYY'] -> ['BHYY', 'BHYY', 'BHAYY']
07. ['SHoHRaT', 'SHHRaT'] -> ['SHOHRT', 'SHOHRAT', 'SHHRT', 'SHHRAT']
08. ['KYAH'] -> ['KYAH']
